# Credit Card Approval Prediction - Model Comparison & Selection

This notebook evaluates **Logistic Regression**, **Decision Tree**, and **Random Forest** models on the preprocessed applicant features, displays comparison tables, plots ROC/Accuracy curves, selects the optimal classifier, and copies dependencies to the Flask Web Application directory.

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

os.makedirs('graphs', exist_ok=True)
print("Libraries loaded.")

Libraries loaded.


In [2]:
df = pd.read_csv(os.path.join('..', '06_Data_Preprocessing', 'processed_dataset.csv'))
X = df.drop(columns=['y'])
y = df['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Features Count: {X.shape[1]}")

Features Count: 18


In [3]:
# Define and fit models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=150, max_depth=8)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model': model,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'AUC': roc_auc_score(y_test, y_prob),
        'y_pred': y_pred,
        'y_prob': y_prob
    }

print("Models training and evaluation completed.")

Models training and evaluation completed.


In [4]:
# Compare Models Table
comparison_list = []
for name in results:
    comparison_list.append({
        'Algorithm': name,
        'Accuracy': results[name]['Accuracy'],
        'Precision': results[name]['Precision'],
        'Recall': results[name]['Recall'],
        'F1 Score': results[name]['F1 Score'],
        'ROC AUC': results[name]['AUC']
    })

comparison_df = pd.DataFrame(comparison_list)
print("=== Performance Comparison Matrix ===")
print(comparison_df.to_string(index=False))
comparison_df.to_csv('model_comparison_metrics.csv', index=False)

=== Performance Comparison Matrix ===
          Algorithm  Accuracy  Precision   Recall  F1 Score  ROC AUC
Logistic Regression     0.823   0.823000 1.000000  0.902907 0.558485
      Decision Tree     0.816   0.822402 0.990279  0.898567 0.520776
      Random Forest     0.823   0.823000 1.000000  0.902907 0.516548


In [5]:
# Plot Accuracy Comparison Graph
plt.figure(figsize=(8, 5))
sns.barplot(x='Algorithm', y='Accuracy', data=comparison_df, palette='Set2')
plt.title('Accuracy Score Comparison of Models', fontsize=14, fontweight='bold')
plt.ylabel('Test Accuracy')
plt.ylim(0, 1.05)
for idx, row in comparison_df.iterrows():
    plt.text(idx, row['Accuracy'] + 0.01, f"{row['Accuracy']*100:.2f}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=300)
plt.savefig('graphs/accuracy_comparison.png', dpi=300)
plt.close()
print("Accuracy comparison plot saved.")

Accuracy comparison plot saved.


In [6]:
# Plot ROC curves comparison
plt.figure(figsize=(9, 7))
colors = {'Logistic Regression': '#2563EB', 'Decision Tree': '#EA580C', 'Random Forest': '#10B981'}
for name in results:
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
    plt.plot(fpr, tpr, color=colors[name], lw=2.5, label=f"{name} (AUC = {results[name]['AUC']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison of Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300)
plt.savefig('graphs/roc_curve.png', dpi=300)
plt.close()
print("ROC curves plot saved.")

ROC curves plot saved.


In [7]:
# Save Best Model and dependencies
# We select the best model based on F1 Score first (and Accuracy if tied) since dataset is imbalanced
best_model_name = comparison_df.sort_values(by=['F1 Score', 'Accuracy'], ascending=False).iloc[0]['Algorithm']
best_clf = results[best_model_name]['model']
print(f"Selected Best Model: {best_model_name}")

joblib.dump(best_clf, 'best_model.pkl')
print("Successfully saved best_model.pkl.")

# Best model confusion matrix
cm = confusion_matrix(y_test, results[best_model_name]['y_pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=['Approved', 'Rejected'], yticklabels=['Approved', 'Rejected'])
plt.title(f'Confusion Matrix of Best Model ({best_model_name})', fontsize=13, fontweight='bold', color='#1E3A8A')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.savefig('graphs/confusion_matrix.png', dpi=300)
plt.close()
print("Best model confusion matrix plot saved.")

Selected Best Model: Logistic Regression
Successfully saved best_model.pkl.


Best model confusion matrix plot saved.


In [8]:
# Propagate files to Flask Web App folder
web_model_dir = os.path.join('..', '08_Web_Application', 'model')
os.makedirs(web_model_dir, exist_ok=True)

# Copy Best Model classifier
shutil.copy2('best_model.pkl', os.path.join(web_model_dir, 'best_model.pkl'))
# Check and copy scaler.pkl and encoder.pkl from preprocessing directory
shutil.copy2(os.path.join('..', '06_Data_Preprocessing', 'scaler.pkl'), os.path.join(web_model_dir, 'scaler.pkl'))
shutil.copy2(os.path.join('..', '06_Data_Preprocessing', 'encoder.pkl'), os.path.join(web_model_dir, 'encoder.pkl'))

# Also save standard 'model.pkl' for compatibility check
shutil.copy2('best_model.pkl', os.path.join(web_model_dir, 'model.pkl'))

print("Propagated machine learning artifacts (best_model.pkl, scaler.pkl, encoder.pkl) to Web Application model repository.")

Propagated machine learning artifacts (best_model.pkl, scaler.pkl, encoder.pkl) to Web Application model repository.
